# Evaluation on CAGE-2: worst-case return and impacted fraction (Figure 2)
Every frozen defender plays the same 200 pre-drawn contexts against four admissible attackers:

1. `Trained P2`, the trained red,
2. `Critic-BR P2`, a myopic best response to the defender's current mixed action under the critic,
3. `Equilibrium P2`, the equilibrium column of the critic's stage game, and
4. `BR-PPO P2`, a recurrent-PPO best response trained from scratch against that defender (`best_response.ipynb`).

Red's head start is played by the trained red for every attacker. Returns are the defender's discounted physical payoff `Reward2` (γ = 0.98, 50 rounds), so attackers minimize them. A defender's worst-case return $\widehat W$ is its lowest mean over the attackers.

Under that worst-case attacker, the **impacted fraction** is the target-weighted fraction of rounds in which Impact is active on an enterprise host, and the **containment rate** is the share of episodes with no Impact at all. Every defender samples its action from its own distribution, as at deployment.

In [ ]:
%run Initialize.ipynb

## Settings, trained models, and the shared contexts

In [ ]:
GAMMA = 0.98
num_episodes = 200
SAVE_DIR = "compared"

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(Player1Actor().to(device), "player1.pth")
Player2       = Load(AttackerActor().to(device), "player2.pth")
trained_red   = red_policy(Player2)
BASELINES     = Load_Baselines(SAVE_DIR)

episode_configs = []
cfg_rng = np.random.RandomState(12345)
for _ in range(num_episodes):
    w = np.array([0.0, 0.0])
    while w.sum() <= 0.0:
        w = cfg_rng.randint(0, 2, size=2).astype(np.float32)
    w = w / w.sum()
    p = np.ones(2, dtype=np.float32)
    while all(p[i] == 1.0 for i in range(2) if w[i] > 0):
        for i in range(2):
            p[i] = cfg_rng.choice([0.0, 0.5, 1.0])
    h = int(cfg_rng.randint(0, 6))
    Theta = np.array([p[0], p[1], w[0], w[1], float(h)], dtype=np.float32)
    episode_configs.append((Theta, Theta[2:Len_Theta].copy(), w, p, h))

## Attackers (`BRAttacker` is defined in `Initialize.ipynb`)

In [ ]:
BR_CACHE = {}
def _load_br(def_name):
    if def_name not in BR_CACHE:
        ck = torch.load(os.path.join(SAVE_DIR, "BR_attacker_" + def_name.replace(" ", "_") + ".pth"),
                        map_location=device, weights_only=True)
        att = BRAttacker(rnn_h=ck['config']['RNN_H']).to(device)
        att.load_state_dict(ck['attacker']); att.eval()
        BR_CACHE[def_name] = att
    return BR_CACHE[def_name]


class BRAttackerWrapper:
    def __init__(self, def_name):
        self.att = _load_br(def_name)
    def reset(self, Theta):
        self.h = None
        self.u_prev = np.zeros(n_u, dtype=np.float32)
        self.theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
    def act(self, red_state):
        obs = np.concatenate([red_state, self.u_prev]).astype(np.float32)
        with torch.no_grad():
            logits, _, self.h = self.att(torch.tensor(obs, device=device).view(1, 1, -1), self.theta_t, self.h)
        u = int(Categorical(logits=logits.view(-1)).sample())
        self.u_prev = np.zeros(n_u, dtype=np.float32); self.u_prev[u] = 1.0
        return u


def attacker_action(Attacker, a_dist, critic_Q, red_state, Type_Theta):
    # critic_Q() returns the defender's payoff matrix under the critic, computed only when needed
    if Attacker == 'Trained P2':
        return trained_red(red_state, Type_Theta)
    Q = critic_Q()
    if Attacker == 'Critic-BR P2':
        return int(np.argmin(a_dist @ Q))            # minimizes the defender's expected payoff
    if Attacker == 'Equilibrium P2':
        _, _, q = MatrixValue(Q)
        q = np.clip(q, 0, None); q = q / q.sum()
        return int(np.random.choice(n_u, p=q))
    raise ValueError(Attacker)

## Rollout

In [ ]:
def run_episodes(Defender, Attacker):
    defw = get_defender(Defender)
    br = BRAttackerWrapper(Defender) if Attacker == 'BR-PPO P2' else None
    ep_returns, ep_impact = [], []
    for i_episode in tqdm(range(num_episodes), desc=f"{Defender} vs {Attacker}", leave=False):
        np.random.seed(1000 + i_episode); torch.manual_seed(1000 + i_episode); random.seed(1000 + i_episode)
        Theta, Type_Theta, w, p_exploit, h = episode_configs[i_episode]
        blue_state, _, red_state, red_knowledge, real_state = Reset_Episode()
        red_knowledge, red_state, real_state = head_start(trained_red, h, red_knowledge, red_state, Type_Theta, p_exploit)
        defw.reset(blue_state, Theta, Type_Theta)
        if br is not None:
            br.reset(Theta)
        # The critic's view of the episode, used by the critic-based attackers
        Context = np.zeros([max_steps, CTX_DIM], dtype=np.float32)
        th_t = torch.tensor(Theta, dtype=torch.float32, device=device)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        disc_sum = 0.0
        impacted_steps = np.zeros(2, dtype=np.float32)
        for t in range(max_steps):
            def critic_Q():
                ctx = torch.tensor(Context, dtype=torch.float32, device=device).view(1, max_steps, CTX_DIM)
                with torch.no_grad():
                    Emb = Context_Model(ctx, Context_Mask(t)).squeeze(0)
                    return critic(torch.tensor(blue_state, dtype=torch.float32, device=device),
                                  torch.tensor(red_state, dtype=torch.float32, device=device),
                                  torch.tensor(real_state, dtype=torch.float32, device=device),
                                  Emb, tt_t, th_t).cpu().numpy().reshape(n_a, n_u)

            a_dist = defw.act_dist(red_state, real_state)
            u = br.act(red_state) if br is not None else attacker_action(Attacker, a_dist, critic_Q, red_state, Type_Theta)
            a = int(np.random.choice(n_a, p=a_dist))         # the defender samples from its own policy
            _, red_knowledge, next_red_state, _ = red_step(u, red_knowledge, red_state, p_exploit)
            blue_action_name, y, next_real_state = blue_step(a)
            defw.observe(a, blue_action_name, y, t)
            next_blue_state = blue_state_vector(y, blue_state, blue_action_name, t + 1)
            for i in range(2):
                if w[i] > 0 and next_real_state[6 + i] == 1:
                    impacted_steps[i] += 1
            disc_sum += (GAMMA ** t) * Reward2(next_real_state, w)
            Context[t, :] = Context_Row(blue_state, a, next_blue_state)
            blue_state, red_state, real_state = next_blue_state, next_red_state, next_real_state
        ep_returns.append(disc_sum)
        ep_impact.append(np.dot(w, impacted_steps) / max_steps)
    return np.array(ep_returns), np.array(ep_impact)

## Evaluate every defender against every attacker

In [ ]:
ATTACKERS = ['Trained P2', 'Critic-BR P2', 'Equilibrium P2', 'BR-PPO P2']
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'MMD']

all_runs = {}
for Defender in DEFENDERS:
    all_runs[Defender] = {Att: run_episodes(Defender, Att) for Att in ATTACKERS}
    print(Defender, {k: round(v[0].mean(), 3) for k, v in all_runs[Defender].items()})

## Worst-case return, impacted fraction, and containment rate

In [ ]:
worst_case = {}
for Defender in DEFENDERS:
    runs = {k: v[0] for k, v in all_runs[Defender].items()}
    worst_att = min(runs, key=lambda k: runs[k].mean())      # attackers minimize the defender's return
    W, impact = runs[worst_att], all_runs[Defender][worst_att][1]
    worst_case[Defender] = dict(W=W.mean(), W_se=W.std(ddof=1) / np.sqrt(len(W)),
                                impact=impact.mean(), impact_se=impact.std(ddof=1) / np.sqrt(len(impact)),
                                contained=(impact == 0).mean(), worst=worst_att)
W_oracle = worst_case['Critic P1']['W']

print(f"{'Defender':<12}{'worst attacker':>16}{'W':>10}{'SE':>8}{'W / W_Oracle':>14}{'Impacted':>10}{'SE':>8}{'Contained':>11}")
print("-" * 89)
for Defender in DEFENDERS:
    c = worst_case[Defender]
    print(f"{Defender:<12}{c['worst']:>16}{c['W']:>10.3f}{c['W_se']:>8.3f}{c['W'] / W_oracle:>13.0%}"
          f"{c['impact']:>10.3f}{c['impact_se']:>8.3f}{c['contained']:>10.1%}")

## Figure 2

In [ ]:
from matplotlib.patches import Patch

fig_labels = ['Oracle', 'Ours', 'R-PPO', 'DR-RPPO', 'VariBAD', 'MMD']
colors     = ['#2ca02c', '#1f77b4', '#d62728', '#ff7f0e', '#9467bd', '#8c564b']
x = np.arange(len(DEFENDERS))

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, key, label in ((ax_left, 'W', 'Worst-case return'), (ax_right, 'impact', 'Impacted fraction')):
    means = [worst_case[d][key] for d in DEFENDERS]
    ses = [worst_case[d][key + '_se'] for d in DEFENDERS]
    ax.bar(x, means, yerr=ses, color=colors, edgecolor='black', linewidth=0.6, capsize=4, width=0.87)
    ax.set_xticks([])
    ax.tick_params(axis='y', labelsize=17)
    ax.set_ylabel(label, fontsize=22)
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
    ax.set_axisbelow(True)
    ax.set_ylim(0, 1.15 * max(m + s for m, s in zip(means, ses)))
    ax.set_xlim(-0.55, len(DEFENDERS) - 0.45)

legend_handles = [Patch(facecolor=c, edgecolor='black', label=l) for c, l in zip(colors, fig_labels)]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.00), ncol=len(fig_labels), fontsize=15, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig('cage2.png', dpi=300, bbox_inches='tight')
plt.show()